# Google TimesFM 3.0 - Python example

Companion notebook for the note **Google TimesFM 3.0**.

- Univariate zero-shot forecast
- Multivariate forecast with covariates (past-only and past+future) on `byd_dolphin_sales.csv`

## Install

In [ ]:
%pip install "timesfm[torch]" ipywidgets jupyter -q

## Load the model

In [ ]:
import numpy as np
import torch
from timesfm3 import TimesFM3Evaluator, ModelConfig

config = ModelConfig(
    checkpoint_path="google/timesfm-3.0-pytorch",
    per_core_batch_size=32,
    device="cpu"
)
forecaster = TimesFM3Evaluator(config)

## Multivariate forecast with covariates

Dataset: `byd_dolphin_sales.csv`, 100 days of simulated BYD Dolphin sales.

| Role | Columns | Shape |
|------|---------|-------|
| Multivariate target | `units_sold`, `test_drives` | `(2, context_length)` |
| Past-only covariates | `website_visits` (only observed up to today) | `(1, context_length)` |
| Past + future covariates | `is_holiday`, `is_weekend`, `is_summer`, weekday sin/cos (calendar, known ahead) | `(5, context_length + horizon)` |

The last `horizon` days are held out to compare the forecast against the actual values.

In [ ]:
import pandas as pd

df = pd.read_csv("byd_dolphin_sales.csv", parse_dates=["date"])

horizon = 21
context_length = len(df) - horizon  # 86 days of history

# Calendar features are known in advance, so they can be encoded for the whole period
dow = df["date"].dt.dayofweek
calendar = pd.DataFrame({
    "is_holiday": df["is_holiday"].astype(float),
    "is_weekend": df["is_weekend"].astype(float),
    "is_summer": (df["season"] == "Summer").astype(float),
    "weekday_sin": np.sin(2 * np.pi * dow / 7),
    "weekday_cos": np.cos(2 * np.pi * dow / 7),
})

history = df.iloc[:context_length]
actual = df.iloc[context_length:]
target_cols = ["units_sold", "test_drives"]

target = history[target_cols].to_numpy(np.float32).T                          # (2, 86)
past_only_cov = history[["website_visits"]].to_numpy(np.float32).T            # (1, 86)
past_future_cov = calendar.to_numpy(np.float32).T                             # (5, 100)

print(target.shape, past_only_cov.shape, past_future_cov.shape)
df.head()

In [ ]:
outputs = list(
    forecaster.predict_batch(
        contexts=[target],
        horizon=horizon,
        past_only_covariates=[past_only_cov],
        past_future_covariates=[past_future_cov],
        return_quantiles=True,
        use_symmetric_averaging=False,
    )
)

forecast = outputs[0].forecast    # (2, 14)     one forecast per variate
quantiles = outputs[0].quantiles  # (2, 14, 9)  quantiles 0.1 ... 0.9
print(forecast.shape, quantiles.shape)

result = pd.DataFrame({"date": actual["date"].dt.date, "weekday": actual["weekday"]})
for i, col in enumerate(target_cols):
    result[f"{col}_actual"] = actual[col].to_numpy()
    result[f"{col}_forecast"] = forecast[i].astype(float).round(1)
    mae = np.abs(forecast[i] - actual[col].to_numpy()).mean()
    print(f"{col}: MAE = {mae:.2f}")
result

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(target_cols), 1, figsize=(11, 7), sharex=True)
for i, (ax, col) in enumerate(zip(axes, target_cols)):
    ax.plot(history["date"], history[col], color="gray", label="history")
    ax.plot(actual["date"], actual[col], color="black", marker="o", ms=3, label="actual")
    ax.plot(actual["date"], forecast[i], color="tab:blue", label="forecast")
    ax.fill_between(actual["date"], quantiles[i, :, 0], quantiles[i, :, -1],
                    color="tab:blue", alpha=0.2, label="10%-90% interval")
    ax.set_title(f"BYD Dolphin - {col}")
    ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## Notes

- Output is a **point forecast + 9 quantiles**, useful for prediction intervals.
- **Zero-shot**: works without fitting on your data; compare against a classic baseline such as ARIMA.
- Outputs are stripped on commit (`nbstripout`), so re-run the cells to see results.